<a href="https://colab.research.google.com/github/saranoor/chatGPT/blob/master/policy_3_train_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install openai

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [12]:
import openai
import pandas as pd
df = pd.read_excel('/content/policy_embeddings_with_qs_ans.xlsx')
olympics_search_fileid = "file-c3shd8wqF3vSCKaukW4Jr1TT"
df.head()

,Unnamed: 0,Unnamed: 0.1,title,heading,content,combined,n_tokens,embeddings,context,questions,answers
0,0,0,10Pearls Policies and Procedures,OUR MISSION,TenPearls (Private) Limited is founded on the ...,Title: 10Pearls Policies and Procedures; Headi...,106,"[-0.0033130680676549673, -0.02280309423804283,...",10Pearls Policies and Procedures\nOUR MISSION\...,1. What is TenPearls' mission?\n2. What are th...,1. TenPearls' mission is to generate opportuni...
1,1,1,Definitions in 10Pearls Policies and Procedures,DEFINITIONS,In this Manual the following words and express...,Title: Definitions in 10Pearls Policies and Pr...,39,"[-0.021558966487646103, -0.0056012338027358055...",Definitions in 10Pearls Policies and Procedure...,"1. What is the definition for ""context""?\n2. W...","1. The definition for ""context"" is the set of ..."
2,2,2,Definitions in 10Pearls Policies and Procedures,Company,“Company” shall mean TenPearls (Private) Limited;,Title: Definitions in 10Pearls Policies and Pr...,28,"[-0.01174114178866148, -0.004361748229712248, ...",Definitions in 10Pearls Policies and Procedure...,"1. What is the definition of ""Company"" in TenP...","1. The definition of ""Company"" in TenPearls Po..."
3,3,3,Definitions in 10Pearls Policies and Procedures,PULSE,“PULSE” shall mean the Company online portal a...,Title: Definitions in 10Pearls Policies and Pr...,34,"[-0.033730316907167435, 0.0022980982903391123,...",Definitions in 10Pearls Policies and Procedure...,1. What is the Company online portal at pulse....,1. The Company online portal at pulse.10pearls...
4,4,4,Definitions in 10Pearls Policies and Procedures,Employee,“Employee(s)” shall mean all employees of the ...,Title: Definitions in 10Pearls Policies and Pr...,51,"[-0.026462620124220848, 0.0019038093741983175,...",Definitions in 10Pearls Policies and Procedure...,1. What is the definition of an employee?\n2. ...,1. The definition of an employee is all employ...


In [13]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
len(train_df), len(test_df)

(112, 29)

### Creating Fine tuning dataset for QnA

In [14]:
import random

def get_random_similar_contexts(question, context, file_id=olympics_search_fileid, search_model='ada', max_rerank=10):
    """
    Find similar contexts to the given context using the search file
    """
    try:
        results = openai.Engine(search_model).search(
            search_model=search_model, 
            query=question, 
            max_rerank=max_rerank,
            file=file_id
        )
        candidates = []
        for result in results['data'][:3]:
            if result['text'] == context:
                continue
            candidates.append(result['text'])
        random_candidate = random.choice(candidates)
        return random_candidate
    except Exception as e:
        print(e)
        return ""

In [16]:
def create_fine_tuning_dataset(df, discriminator=False, n_negative=1, add_related=False):
    """
    Create a dataset for fine tuning the OpenAI model; either for a discriminator model, 
    or a model specializing in Q&A, where it says if no relevant context is found.

    Parameters
    ----------
    df: pd.DataFrame
        The dataframe containing the question, answer and context pairs
    discriminator: bool
        Whether to create a dataset for the discriminator
    n_negative: int
        The number of random negative samples to add (using a random context)
    add_related: bool
        Whether to add the related contexts to the correct context. These are hard negative examples

    Returns
    -------
    pd.DataFrame
        The dataframe containing the prompts and completions, ready for fine-tuning
    """
    rows = []
    for i, row in df.iterrows():
        for q, a in zip(("1." + row.questions).split('\n'), ("1." + row.answers).split('\n')):
            if len(q) >10 and len(a) >10:
                if discriminator:
                    rows.append({"prompt":f"{row.context}\nQuestion: {q[2:].strip()}\n Related:", "completion":f" yes"})
                else:
                    rows.append({"prompt":f"{row.context}\nQuestion: {q[2:].strip()}\nAnswer:", "completion":f" {a[2:].strip()}"})

    for i, row in df.iterrows():
        for q in ("1." + row.questions).split('\n'):
            if len(q) >10:
                for j in range(n_negative + (2 if add_related else 0)):
                    random_context = ""
                    if j == 0 and add_related:
                        # add the related contexts based on originating from the same wikipedia page
                        subset = df[(df.title == row.title) & (df.context != row.context)]
                        
                        if len(subset) < 1:
                            continue
                        random_context = subset.sample(1).iloc[0].context
                    if j == 1 and add_related:
                        # add the related contexts based on the most similar contexts according to the search
                        random_context = get_random_similar_contexts(q[2:].strip(), row.context, search_model='ada', max_rerank=10)
                    else:
                        while True:
                            # add random context, which isn't the correct context
                            random_context = df.sample(1).iloc[0].context
                            if random_context != row.context:
                                break
                    if discriminator:
                        rows.append({"prompt":f"{random_context}\nQuestion: {q[2:].strip()}\n Related:", "completion":f" no"})
                    else:
                        rows.append({"prompt":f"{random_context}\nQuestion: {q[2:].strip()}\nAnswer:", "completion":f" No appropriate context found to answer the question."})

    return pd.DataFrame(rows) 


In [17]:
for name, is_disc in [('discriminator', True), ('qa', False)]:
    for train_test, dt in [('train', train_df), ('test', test_df)]:
        ft = create_fine_tuning_dataset(dt, discriminator=is_disc, n_negative=1, add_related=True)
        ft.to_json(f'{name}_{train_test}.jsonl', orient='records', lines=True)

search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search
search

In [18]:
API_KEY = '' # Add your API key here
openai.api_key = API_KEY

In [19]:
!export OPENAI_API_KEY=API_KEY

In [20]:
!openai api fine_tunes.create -t "discriminator_train.jsonl" -v "discriminator_test.jsonl" --batch_size 16  --compute_classification_metrics --classification_positive_class " yes" --model ada

Error: No API key provided. You can set your API key in code using 'openai.api_key = <API-KEY>', or you can set the environment variable OPENAI_API_KEY=<API-KEY>). If your API key is stored in a file, you can point the openai module at it with 'openai.api_key_path = <PATH>'. You can generate API keys in the OpenAI web interface. See https://onboard.openai.com for details, or email support@openai.com if you have any questions.


In [9]:
!openai api fine_tunes.create -t "qa_train.jsonl" -v "qa_test.jsonl" --batch_size 16

Error: No API key provided. You can set your API key in code using 'openai.api_key = <API-KEY>', or you can set the environment variable OPENAI_API_KEY=<API-KEY>). If your API key is stored in a file, you can point the openai module at it with 'openai.api_key_path = <PATH>'. You can generate API keys in the OpenAI web interface. See https://onboard.openai.com for details, or email support@openai.com if you have any questions.


In [26]:
context="All female Full-time confirmed Employees are entitled to Maternity Leave with pay for a \
period of 6 weeks starting from the date as mutually agreed between the female Employee \
and the Company."

In [28]:
ft_discriminator = "ada:ft-personal-2023-03-13-09-17-44"
ft_qa = "curie:ft-personal-2023-03-13-09-40-39"

def apply_ft_discriminator(context, question, discriminator_model):
    """
    Apply the fine tuned discriminator to a question, to assess whether it can be answered from the context.
    """
    prompt = f"{context}\nQuestion: {question}\n Related:"
    result = openai.Completion.create(model=discriminator_model, prompt=prompt, max_tokens=1, temperature=0, top_p=1, n=1, logprobs=2)
    return result['choices'][0]['logprobs']['top_logprobs']

apply_ft_discriminator(context, 
                        '10Pearls maternity leaves duration', ft_discriminator)

[<OpenAIObject at 0x7f7108ea29f0> JSON: {
   " no": -2.9543774,
   " yes": -0.056348264
 }]

In [30]:
def apply_ft_qa_answer(context, question, answering_model):
    """
    Apply the fine tuned discriminator to a question
    """
    prompt = f"{context}\nQuestion: {question}\nAnswer:"
    result = openai.Completion.create(model=answering_model, prompt=prompt, max_tokens=30, temperature=0, top_p=1, n=1, stop=['.','\n'])
    return result['choices'][0]['text']

apply_ft_qa_answer(context, 
                    '10Pearls maternity leaves duration', ft_qa)

' 10Pearls maternity leaves duration is 6 weeks'